In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, upper, sum, desc, concat_ws, lit, when, trim, coalesce, lower, array_contains, array_distinct, array, concat, filter, datediff, count, avg, length, floor, regexp_extract, regexp_replace, split, element_at, to_date, hour, array_join, slice, struct

In [0]:
# one rule = one dictionary, made by make_check

def make_check(name, columns, bad, action):
    if action not in ACTION_TAG:
        raise ValueError(f"rule '{name}': unknown action '{action}' (use one of {list(ACTION_TAG)})")
    return {"name": name, "columns": columns, "bad": bad, "action": action}

In [0]:
# the rows that would stay in Silver if the run ended now. used for profiles that describe "the data we keep"

def unflagged(df: DataFrame) -> DataFrame:
    return df.filter(UNFLAGGED)

In [0]:
def is_missing(name):
    return col(name).isNull() & ~array_contains(col("_reasons"), f"{name}_unreadable")

In [0]:
def counted_bad(c):
    return c["bad"] if c["action"] == "quarantine" else (c["bad"] & UNFLAGGED)

In [0]:
# how many rows break each rule?

def count_violations(df: DataFrame, checks: list) -> dict:
    row = df.agg(*[sum(counted_bad(c).cast("int")).alias(c["name"]) for c in checks]).collect()[0].asDict()
    return {name: (n or 0) for name, n in row.items()}

In [0]:
# print the count of each rule and show example rows before anything is changed.
# The examples show the original Bronze text 

def show_violations(df: DataFrame, checks: list, topic: str, limit: int = 10) -> dict:
    counts = count_violations(df, checks)
    raw_fields = df.schema["_raw"].dataType.fieldNames()

    for c in checks:
        n = counts[c["name"]]
        if n == 0:
            print(f"[OK][{topic}] {c['name']}: no rows")
            continue

        print(f"[{ACTION_TAG[c['action']]}][{topic}] {c['name']}: {n:,} rows (action: {c['action']}). Examples:")
        shown = [col("_raw.id").alias("raw_id")]
        for column in c["columns"]:
            if column in raw_fields and column != "id":
                shown.append(col(f"_raw.{column}").alias(f"raw_{column}"))
            shown.append(col(column))
        display(df.filter(counted_bad(c)).select(*shown).limit(limit))

    return counts

In [0]:
# apply_nulling: for the rules whose action is "null", set the columns of the rule to NULL where the row breaks it

def apply_nulling(df: DataFrame, checks: list, topic: str) -> DataFrame:
    for c in checks:
        if c["action"] == "null":
            for column in c["columns"]:
                df = df.withColumn(column, when(c["bad"], None).otherwise(col(column)))
            print(f"[NULL][{topic}] {c['name']}: bad values in {c['columns']} set to NULL")
    return df

In [0]:
# for the rules whose action is "quarantine", add the rule name to `_reasons` of every row that breaks it. No row is removed here.

def mark_quarantine(df: DataFrame, checks: list, topic: str) -> DataFrame:
    rules = [(c["name"], c["bad"]) for c in checks if c["action"] == "quarantine"]
    if not rules:
        print(f"[INFO]-[{topic}]: no rule with action 'quarantine'")
        quarantine_counts[topic] = 0
        return df
    
    # the names of the rules this row breaks
    broken_here = filter(
        array(*[when(bad, lit(name)) for name, bad in rules]),
        lambda r: r.isNotNull()
    )
    any_broken = size(broken_here) > 0

    n, n_new = df.agg(
        sum(any_broken.cast("int")),
        sum((any_broken & UNFLAGGED).cast("int"))
    ).collect()[0]
    n, n_new = n or 0, n_new or 0

    quarantine_counts[topic] = n

    print(f"[QUARANTINE]-[{topic}]: {n:,} rows break a quarantine rule of this topic ({n_new:,} of them were not marked by an earlier topic)")
    
    return df.withColumn("_reasons", array_distinct(concat(col("_reasons"), broken_here)))

In [0]:
# after the fix, every "null" rule must have 0 rows left, and every row that breaks a "quarantine" rule must carry the name of that rule in `_reasons`

def verify_checks(df: DataFrame, checks: list, topic: str) -> None:

    to_count = [dict(c, bad=c["bad"] & ~array_contains(col("_reasons"), c["name"])) if c["action"] == "quarantine" else c for c in checks]
    counts = count_violations(df, to_count)

    still_broken = {c["name"]: counts[c["name"]] for c in checks if c["action"] in ("null", "quarantine") and counts[c["name"]] > 0}

    if still_broken:
        raise ValueError(f"[ERROR]-[{topic}][VERIFY]: rules still broken after the fix: {still_broken}")

    warnings = {c["name"]: counts[c["name"]] for c in checks if  c["action"] == "warn" and counts[c["name"]] > 0}
    if warnings:
        print(f"[WARN]-[{topic}][VERIFY]: rows kept as they are: {warnings}")
    else: 
        print(f"[OK]-[{topic}][VERIFY]: every rule passes")